In [2]:
# pip install pandas scikit-learn

import time
import warnings

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Load a churn dataset whose target column is named "Churn"
df = pd.read_csv("telco_churn.csv")
target_col = "Churn"

# Remove an ID column if present; it is not a useful predictive feature
df = df.drop(columns=["customerID"], errors="ignore")

# Convert common churn labels to 0 and 1
y = df.pop(target_col)
if y.dtype == "object":
    y = y.astype(str).str.strip().str.lower().map(
        {"yes": 1, "no": 0, "true": 1, "false": 0}
    )

if y.isna().any():
    raise ValueError("Could not interpret every target value in the Churn column.")

X_train, X_test, y_train, y_test = train_test_split(
    df, y.astype(int), test_size=0.2, random_state=42, stratify=y
)

# Impute missing values, scale numeric features, and encode categories.
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_cols),
])

# sklearn calls sigmoid activation "logistic"
activations = {
    "ReLU": "relu",
    "Sigmoid": "logistic",
    "Tanh": "tanh",
}

results = []

for name, activation in activations.items():
    model = Pipeline([
        ("preprocess", preprocessor),
        ("network", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation=activation,
            solver="adam",
            learning_rate_init=0.001,
            max_iter=300,
            early_stopping=True,
            n_iter_no_change=15,
            random_state=42,
        )),
    ])

    # Show convergence warnings if a model reaches max_iter before stopping
    with warnings.catch_warnings():
        warnings.simplefilter("always", ConvergenceWarning)
        start = time.perf_counter()
        model.fit(X_train, y_train)
        elapsed = time.perf_counter() - start

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    network = model.named_steps["network"]

    results.append({
        "Activation": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Epochs": network.n_iter_,
        "Training time (s)": elapsed,
    })

results_df = pd.DataFrame(results).set_index("Activation")
print(results_df.round(4).to_string())
print("\nBest F1:", results_df["F1"].idxmax())

FileNotFoundError: [Errno 2] No such file or directory: 'telco_churn.csv'